# PosterLlama + HyperCLOVA X 비전 인코더

제공한 2단계 아키텍처를 실행하는 노트북입니다. **기본 `MODE = "smoke"`에서는 대형 가중치 다운로드 없이 모든 셀을 실행할 수 있습니다.** 작은 무작위 SigLIP/Llama 모델로 실제 forward/backward, PEFT LoRA, 저장·복원을 확인합니다. 이 모드의 출력은 학습된 포스터 결과가 아닙니다.

| 단계 | 이미지 인코더 | Adapter | CodeLlama | 목표 |
|---|---|---|---|---|
| 1 | 동결 | 학습 | 동결 | 이미지 → 자연어 설명 |
| 2 | 동결 | 1단계 가중치 동결 | 기본 가중치 동결 + LoRA 학습 | 이미지 + 지시문 → HTML 레이아웃 |

기본 실모델은 `HyperCLOVAX-SEED-Vision-Instruct-3B` 안의 **SigLIP 비전 인코더**와 `CodeLlama-7b-hf`입니다. 독립적인 “HyperCLOVA X CLIP” 체크포인트가 지정되지 않아 공개 모델의 비전 타워를 사용합니다. HyperCLOVA 언어 모델, AnyRes, C-Abstractor는 사용하지 않습니다.

출처: [HyperCLOVA 공식 설정](https://huggingface.co/naver-hyperclovax/HyperCLOVAX-SEED-Vision-Instruct-3B/blob/main/config.json), 로컬 원본 `~/Desktop/car/PosterLlama/src/model/minigpt4.py`, `minigpt_base.py`.

## 0. 커널 및 패키지

Python **3.10 이상**의 커널을 선택하세요. 현재 머신에는 `naver` Conda 환경에 필요한 ML 라이브러리가 설치되어 있습니다. VS Code의 커널 선택에서 해당 환경을 선택할 수 있습니다.

새 환경이라면 아래 `%pip` 줄의 주석을 해제해 설치한 뒤 커널을 재시작하세요. NVIDIA GPU에서는 해당 CUDA 버전의 PyTorch를 먼저 설치하세요. 4-bit 옵션을 사용할 때만 `bitsandbytes`가 추가로 필요합니다.

In [ ]:
import sys
from pathlib import Path
assert sys.version_info >= (3, 10), f"Python 3.10+ 커널이 필요합니다: {sys.version}"
print("Python:", sys.executable)
# %pip install -r requirements.txt
# %pip install bitsandbytes  # CUDA에서 4-bit를 사용할 때만

In [ ]:
import gc
import torch
from PIL import Image
from IPython.display import display

PROJECT_ROOT = Path.cwd()
assert (PROJECT_ROOT / "posterllama_hcx").is_dir(), "노트북 작업 디렉터리를 PosterLlama 프로젝트로 설정하세요."
from posterllama_hcx import (
    ModelConfig, TrainConfig, PosterLlama, PosterDataset,
    train_stage, evaluate, build_layout_prompt, render_layout,
)
from posterllama_hcx.smoke import write_demo_data
from posterllama_hcx.training import seed_everything

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available(), "MPS:", torch.backends.mps.is_available())

## 1. 모드와 데이터 경로

`smoke`는 합성 이미지 1장으로 각 단계 1 optimizer step만 수행합니다. **실제 모델을 실행하려면 `MODE = "real"`로 바꾸고 아래 경로를 지정하세요.**

1단계 JSONL: `{"image": "images/a.jpg", "caption": "제품 이미지 설명"}`

2단계 JSONL: `{"image": "images/a.jpg", "prompt": "작업 지시와 텍스트 조건", "html": "<html>...<svg>...<rect .../></svg>...</html>"}`

이미지 경로는 `IMAGE_ROOT` 기준입니다. `None`이면 각 JSONL 파일의 부모 폴더 기준입니다. 원본 논문의 `name` 및 `*_seq_modeling` + `<MID>` 데이터도 읽을 수 있습니다. 원본 1단계 WebDataset tar는 직접 읽지 않으므로 이미지/캡션 JSONL로 준비하세요. 실학습에서는 별도의 검증 데이터를 지정하세요.

In [ ]:
MODE = "smoke"  # "real"로 바꾸면 실제 HyperCLOVA 비전 타워 + CodeLlama를 로드합니다.
assert MODE in {"smoke", "real"}

STAGE1_JSONL = PROJECT_ROOT / "data/stage1/train.jsonl"
STAGE2_JSONL = PROJECT_ROOT / "data/stage2/train.jsonl"
STAGE1_VAL_JSONL = None
STAGE2_VAL_JSONL = None
IMAGE_ROOT = None
PAPER_TASK = "unconditional"  # 원본 데이터: "random", "cond_cate", "completion" 등
TRAIN_STAGE1 = True           # False이면 아래 stage1 체크포인트를 불러옵니다.
TRAIN_STAGE2 = True           # False이면 아래 stage2 체크포인트로 추론합니다.

RUN_DIR = PROJECT_ROOT / "outputs" / ("smoke_notebook" if MODE == "smoke" else "hcx_codellama")
STAGE1_CHECKPOINT = RUN_DIR / "stage1"
STAGE2_CHECKPOINT = RUN_DIR / "stage2"
RUN_DIR.mkdir(parents=True, exist_ok=True)

if MODE == "smoke":
    torch.set_num_threads(2)
    demo = write_demo_data(RUN_DIR / "demo_data")
    STAGE1_JSONL, STAGE2_JSONL = demo / "stage1.jsonl", demo / "stage2.jsonl"
    IMAGE_ROOT = demo

# 데이터 오류는 대형 모델 다운로드 전에 확인합니다.
stage1_data = PosterDataset(STAGE1_JSONL, stage=1, image_root=IMAGE_ROOT) if TRAIN_STAGE1 else None
stage2_data = PosterDataset(STAGE2_JSONL, stage=2, image_root=IMAGE_ROOT, task=PAPER_TASK)
val1 = PosterDataset(STAGE1_VAL_JSONL, stage=1, image_root=IMAGE_ROOT) if STAGE1_VAL_JSONL else None
val2 = PosterDataset(STAGE2_VAL_JSONL, stage=2, image_root=IMAGE_ROOT, task=PAPER_TASK) if STAGE2_VAL_JSONL else None
if stage1_data is not None:
    _ = stage1_data[0]
example = stage2_data[0]
display(example["image"])
print("입력 지시문:", example["prompt"])
print("데이터 수:", len(stage1_data) if stage1_data is not None else "stage1 재사용", len(stage2_data))

## 2. 모델 로드

실모델에서는 HCX의 비전 가중치가 포함된 shard(현재 약 5 GB)와 CodeLlama 가중치를 처음 한 번 다운로드합니다. HCX 언어 모델은 메모리에 로드하지 않습니다. 모델 접근에 인증이 필요하면 터미널에서 `huggingface-cli login`을 사용하세요.

- 권장 실학습 환경: NVIDIA CUDA GPU. 메모리는 시퀀스 길이, 배치 크기, 정밀도에 따라 달라집니다.
- 메모리가 부족하면 `load_in_4bit=True`, 배치 1, gradient accumulation을 사용하세요. 이는 양자화 기본 모델에 LoRA를 학습하는 변형입니다.
- CPU/Mac에서는 먼저 `smoke` 모드로 확인하세요. `smoke`는 항상 CPU float32입니다.
- 378×378 전체 이미지 한 장을 letterbox로 입력합니다. 729개 패치를 4개씩 묶고 마지막 그룹을 0으로 채워 **183 visual tokens**를 만듭니다. 텍스트 토큰은 CodeLlama의 tokenizer/embedding을 사용합니다.
- 원본 DINO Adapter 체크포인트는 HCX Adapter와 호환되지 않습니다. HCX 기준으로 1단계를 다시 학습해야 합니다.

In [ ]:
model_config = ModelConfig(
    smoke=(MODE == "smoke"),
    vision_model="naver-hyperclovax/HyperCLOVAX-SEED-Vision-Instruct-3B",
    language_model="codellama/CodeLlama-7b-hf",  # 로컬 모델 경로도 가능
    device="auto",
    dtype="auto",  # CUDA bf16 지원 시 bfloat16, 그 외 float32
    load_in_4bit=False,
    gradient_checkpointing=True,
    lora_r=4 if MODE == "smoke" else 64,
    max_prompt_tokens=1024,
    max_target_tokens=1024,
)
seed_everything(model_config.seed)
model = PosterLlama.from_config(model_config)
print(model.parameter_summary())
print("1단계 학습 대상:", model.assert_trainable_parameters())

## 3. 1단계 — 이미지 설명으로 Adapter 학습

정답 caption과 EOS에만 causal language modeling loss를 적용합니다. 이미지·지시문·padding 토큰의 label은 `-100`입니다. LLM 가중치는 동결하되 LLM 연산을 통과하는 gradient는 유지해서 Adapter까지 전달합니다.

아래 실학습 설정은 시작용 예시입니다. 에폭 수와 학습률은 데이터 규모에 맞춰 조정하세요. `max_steps=None`이면 지정한 에폭 전체를 실행합니다. 체크포인트는 매 에폭 끝에 저장되며 optimizer/scheduler 상태를 포함한 중단 지점 복구는 지원하지 않습니다.

In [ ]:
stage1_train_config = TrainConfig(
    epochs=1,
    batch_size=1,
    gradient_accumulation_steps=1 if MODE == "smoke" else 4,
    learning_rate=1e-4,
    max_steps=1 if MODE == "smoke" else None,
)
if TRAIN_STAGE1:
    stage1_history = train_stage(model, stage1_data, stage1_train_config,
                                 STAGE1_CHECKPOINT, validation=val1)
    print(stage1_history[-1])
else:
    model.load_stage1_adapter(STAGE1_CHECKPOINT)
print("1단계 체크포인트:", STAGE1_CHECKPOINT)

## 4. 2단계 — HTML 레이아웃으로 CodeLlama LoRA 학습

저장한 1단계 Adapter를 읽고 동결합니다. CodeLlama의 `q_proj`, `v_proj`에 LoRA를 붙여 LoRA 파라미터만 학습합니다. HTML을 토큰 제한에서 잘라내지 않으며 길이가 초과되면 오류로 알려줍니다.

In [ ]:
model.load_stage1_adapter(STAGE1_CHECKPOINT)
model.start_stage2()
print(model.parameter_summary())
print("2단계 학습 파라미터 예:", model.assert_trainable_parameters()[:6])

stage2_train_config = TrainConfig(
    epochs=1,
    batch_size=1,
    gradient_accumulation_steps=1 if MODE == "smoke" else 4,
    learning_rate=1e-4,
    max_steps=1 if MODE == "smoke" else None,
)
if TRAIN_STAGE2:
    stage2_history = train_stage(model, stage2_data, stage2_train_config,
                                 STAGE2_CHECKPOINT, validation=val2)
    print(stage2_history[-1])
print("2단계 체크포인트:", STAGE2_CHECKPOINT)

## 5. 저장된 모델 복원 및 추론

기본 가중치를 중복해서 메모리에 올리지 않도록 학습 모델을 해제한 후 Adapter/LoRA를 복원합니다. 아래 예시는 **학습 데이터 중 한 장에 대한 동작 확인**입니다. 성능 평가는 별도의 검증/테스트 이미지로 진행하세요.

In [ ]:
del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
if torch.backends.mps.is_available():
    torch.mps.empty_cache()

model = PosterLlama.from_checkpoint(STAGE2_CHECKPOINT, gradient_checkpointing=False)
pixels = model.image_processor([example["image"]])
generated_html = model.generate(
    pixels, [example["prompt"]], max_new_tokens=24 if MODE == "smoke" else 512,
)[0]
(RUN_DIR / "generated.txt").write_text(generated_html, encoding="utf-8")
print(generated_html)
if MODE == "smoke":
    print("위 출력은 무작위 소형 모델의 결과입니다. 정상 HTML 또는 유효한 포스터 품질을 기대하지 않습니다.")

## 6. 좌표 시각화

정답 레이아웃 미리보기와 모델 출력을 구분합니다. 렌더러는 `<svg>/<rect>` 좌표만 해석하고 원본 이미지에 박스를 표시합니다. 실제 홍보 문구의 폰트·색상·타이포그래피 배치는 별도 작업입니다. 생성 HTML이 잘못되거나 좌표가 캔버스를 벗어나면 오류를 표시합니다.

In [ ]:
print("정답 레이아웃 (모델 출력 아님)")
display(render_layout(example["image"], example["target"], RUN_DIR / "ground_truth.png"))
if MODE == "real":
    try:
        preview = render_layout(example["image"], generated_html, RUN_DIR / "generated_layout.png")
    except ValueError as error:
        print("생성 레이아웃 검증 실패:", error)
        print("generated.txt를 확인하세요. 학습량 또는 max_new_tokens 조정이 필요할 수 있습니다.")
    else:
        print("모델 생성 레이아웃")
        display(preview)

## 7. 내 이미지와 홍보 문구로 실행 (선택)

실학습을 마친 후 `CUSTOM_IMAGE`에 제품/배경 이미지 경로를 넣으세요. 원본 포스터의 글자가 제거된 이미지가 레이아웃 학습에 적합합니다. 이미지가 바뀌면 캔버스 크기도 실제 이미지 크기를 사용합니다.

In [ ]:
CUSTOM_IMAGE = None  # 예: PROJECT_ROOT / "data/my_product.jpg"
PROMO_TEXTS = ["신제품 출시", "지금 구매하면 20% 할인"]

if CUSTOM_IMAGE is not None:
    if MODE != "real":
        raise ValueError("내 이미지 추론에는 실학습된 모델이 필요합니다. MODE='real'을 사용하세요.")
    with Image.open(CUSTOM_IMAGE) as source:
        custom_image = source.convert("RGB")
    prompt = build_layout_prompt(PROMO_TEXTS, width=custom_image.width, height=custom_image.height)
    result = model.generate(model.image_processor([custom_image]), [prompt], max_new_tokens=768)[0]
    print(result)
    (RUN_DIR / "custom_generated.txt").write_text(result, encoding="utf-8")
    display(render_layout(custom_image, result, RUN_DIR / "custom_layout.png"))